In [1]:
import sys
import os
import random

import json
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

sys.path.append(f"{os.environ['MDS']}/code")
from project_utils import load_generations
from project_utils import load_config


In [2]:
experiments_config = load_config(f"{os.environ['MDS']}/code/experiments_mayarr.json")
experiment_results_folder = f"{os.environ['MDS']}/results_mayarr_sample"

# Prepare Data

In [3]:
df = []
split = "test"
pbar = tqdm(total=len(experiments_config["datasets"]) * len(experiments_config["models"]))
for dataset in experiments_config["datasets"]:
    ground_truth_path = f"{os.environ['MDS']}/data/{dataset}/{split}.jsonl"
    if "_sample" in experiment_results_folder:
        ground_truth_path = ground_truth_path.replace(".jsonl", ".sample.jsonl")
        
    for model in experiments_config["models"]:
#         if model == "ground_truth":
#             continue
        
        # Prepare paths
        results_base = f"{experiment_results_folder}/{model}/{dataset}/{split}"
        generated_path = os.path.join(results_base, "generated.jsonl")
        results_path = os.path.join(results_base, "evaluation.json")
        
        if not os.path.exists(results_path):
            print(f"{results_path} does not exist. Skipping.")
            continue
        
        # Load generations and ground truth
        temp = [e for e in load_generations(ground_truth_path, generated_path)]
        temp = pd.DataFrame(temp)
        temp["model"] = model
        temp["dataset"] = dataset
        temp["n_docs"] = temp.source.map(len)
        
        # Load results
        with open(results_path, "r") as f:
            results = json.load(f)
        
        for metric_name, metric_results in results.items():
            metric_results = metric_results.get("individual", [pd.NA] * len(temp))
            
            try:
                temp[metric_name] = metric_results
            except ValueError as err:
                print(f"Error loading {results_path}. Skipping. \n{err=}")
                continue
        
        df.append(temp)
        pbar.update(1)
pbar.close()

# Cleanup
df = pd.concat(df).reset_index(drop=True)

100%|██████████| 42/42 [00:04<00:00, 10.38it/s]


In [4]:
def unroll_dict(row):
    new_cols = {}
    for metric in ["rouge", "unieval", "compression", "prometheus"]:
        if isinstance(row[metric], dict):
            for sub_metric, v in row[metric].items():
                new_cols[f"{metric}_{sub_metric}"] = v
    return new_cols

unrolled_cols = df.apply(func=unroll_dict, axis="columns", result_type="expand")
df = df.join(unrolled_cols)

In [5]:
# Drop examples without Prometheus
df.dropna(subset=["prometheus_fluency"], inplace=True)

In [6]:
df.iloc[2]

id                        https://www.reddit.com/r/buildapc/comments/zmb...
source                    [Title: [Build Help] I need a decision now and...
target                    For building a pc, commenters suggest going fo...
generated                 For building a pc, commenters suggest going fo...
prob                                                                    1.0
error                                                                  None
model                                                          ground_truth
dataset                                                    convosumm/reddit
n_docs                                                                   12
compression               {'sentence': 0.043478260869565216, 'token': 0....
moverscore                                                              1.0
prometheus                {'coherence': 3, 'consistency': 3, 'coverage':...
rouge                     {'rouge1': 1.0, 'rouge2': 1.0, 'rougeL': 1.0, ...
unieval     

In [7]:
def filter_row(row):
    if "PRIMERA" not in row.model:
        return False
    if row.dataset != "multixscience":
        return False
    return True


for idx, ex in df[df.apply(filter_row, axis="columns")].sample(n=5, random_state=42).iterrows():
    print(f"""{idx=} {ex.model=} 
    {ex.compression_sentence=:.2} {ex.compression_token=:.2} {ex.rouge_rouge1=:.2}
    GENERATED: {ex.generated=}
    
    TARGET: {ex.target}
    
    SOURCE: {ex.source=}
    -------------------
    """)

idx=1703 ex.model='PRIMERA-MultiNews_Plus' 
    ex.compression_sentence=6.2 ex.compression_token=5.1 ex.rouge_rouge1=0.3
    GENERATED: ex.generated='The Pascal Visual Object Classes (VOC) challenge is a benchmark in visual object category recognition and detection, providing the vision and machine learning communities with a standard dataset of images and annotation, and standard evaluation procedures. Organised annually from 2005 to present, the challenge and its associated dataset has become accepted as the benchmark for object detection. This paper describes the dataset and evaluation procedure. We review the state-of-the-art in evaluated methods for both classification and detection of both classification methods, analyse whether the methods are statistically different, what they are learning from the images (e.g. the objects or their context), and what the methods find easy or confuse, and propose directions for future improvement and extension..'
    
    TARGET: In the recent y

In [8]:
df.dataset.unique()

array(['convosumm/reddit', 'multixscience', 'multinews_plus'],
      dtype=object)

# Look at Data

Generations from ALL models for a SINGLE example

In [11]:
all_ids = df[df.dataset=="convosumm/reddit"].id.unique()
random.seed(20)
random_id = random.choice(all_ids)

# ex_id = random_id
ex_id = "https://www.reddit.com/r/elderscrollsonline/comments/2urzo7/what_eso_going_sub_free/"

example = df[df.id==ex_id]
source_docs = "\n---------------\n".join(example.iloc[0].source)
show_metrics = ["compression_token", "rouge_rougeLsum", "prometheus_factuality", "unieval_factuality", "prometheus_fluency", "unieval_fluency", "prometheus_relevance", "unieval_relevance"]

print(f"""Showing generations for {ex_id=}.
TARGET: {example.iloc[0].target}

SOURCE:
{source_docs}
------------------------------
""")

for row_id, ex in example.iterrows():
    out = f"""MODEL: {ex.model}

GENERATED: {ex.generated}
"""

    for m in show_metrics:
        out += f"{m}: {ex[m]:.2f}\t"
    out += "\n------\n"
    print(out)


Showing generations for ex_id='https://www.reddit.com/r/elderscrollsonline/comments/2urzo7/what_eso_going_sub_free/'.
TARGET: A couple of the responses to this post are either satirical or sarcastic. One commenter states that ESO is going B2P beginning on March 17, 2015. This is a rumor, and it seems as though mixed information is being spread about this event. Some commenters believe it has to do with boosting sales, difficulties working with Microsoft, or additional access to games.

SOURCE:
Title: What! ESO going sub free? Subreddit: elderscrollsonline When did this happend? Haven't played since late September. 
---------------
ESO is going B2P starting March 17th 2015.
---------------
Why did they make that decision?
---------------
No official word, heard it had to do with Xbox not allowing subscription games on it (where as PS4 has a history of allowing it).  It launches on consoles June 9th.
---------------
They both allow sub based games, the issue is that console users would p

Specific model / dataset

In [ ]:
model = "allenai/PRIMERA-multixscience"
dataset = "multinews"
metric = "rouge_rouge1"
N = 100
for row_id, row in df[(df.model==model)&(df.dataset==dataset)].head(N).iterrows():
    print(f"""{row.id} {metric}={row[metric]:.2} Model={row.model} Dataset={row.dataset} #Articles={len(row.source)}
    Generated: {row.generated}
    
    Target: {row.target}
    ------------
    """)

Look by metric

In [ ]:
metric = "moverscore"
N = 100
for row_id, row in df.sort_values(by=metric, ascending=True).head(N).iterrows():
    print(f"""{row.id} {metric}={row[metric]:.2} Model={row.model} Dataset={row.dataset} #Articles={len(row.source)}
    Generated: {row.generated}
    
    Target: {row.target}
    ------------
    """)


Look at a specific model

In [ ]:
for row_id, row in df[df.model=="ICL-multixscience"].sample(n=5, random_state=42).iterrows():
    sources = "\n\nDoc: ".join(row.source)
    print(f"""{row.id}
    
    {sources}
    -----------
    Generated: {row.generated}
    
    Target: {row.target}
    
    """)